In [0]:
dqc_queries = {
    "Trips - Check for Null VendorID": """
SELECT
VendorID
FROM workspace.taxi_weather.silver_trips
WHERE VendorID IS NULL
""",
    "Trips - Check for Unknown VendorID": """
SELECT DISTINCT
bt.VendorID
FROM workspace.taxi_weather.silver_trips AS bt
LEFT ANTI JOIN workspace.taxi_weather.silver_vendor_dim AS sv
ON bt.VendorID = sv.VendorID
""",
    "Trips - Check for Null RatecodeID": """
SELECT
RatecodeID
FROM workspace.taxi_weather.silver_trips
WHERE RatecodeID IS NULL
""",
    "Trips - Check for Unknown RatecodeID": """
SELECT DISTINCT
bt.RatecodeID
FROM workspace.taxi_weather.silver_trips AS bt
LEFT ANTI JOIN workspace.taxi_weather.silver_rate_dim AS sr
ON bt.RatecodeID = sr.RatecodeID
""",
    "Trips - Check for Null Store and Forward Flag": """
SELECT
store_and_fwd_flag
FROM workspace.taxi_weather.silver_trips
WHERE store_and_fwd_flag IS NULL
""",
    "Trips - Check for Untrimmed Store and Forward Flag": """
SELECT
store_and_fwd_flag
FROM workspace.taxi_weather.silver_trips
WHERE store_and_fwd_flag != TRIM(store_and_fwd_flag)
""",
    "Trips - Check for Null Payment Type": """
SELECT
payment_type
FROM workspace.taxi_weather.silver_trips
WHERE payment_type IS NULL
""",
    "Trips - Check for Unknown Payment Type": """
SELECT DISTINCT
bt.payment_type
FROM workspace.taxi_weather.silver_trips AS bt
LEFT ANTI JOIN workspace.taxi_weather.silver_payment_dim AS sp
ON bt.payment_type = sp.payment_type
""",
    "Trips - Check for Null Pickup LocationID": """
SELECT
PULocationID
FROM workspace.taxi_weather.silver_trips
WHERE PULocationID IS NULL
""",
    "Trips - Check for Unknown Pickup LocationID": """
SELECT DISTINCT
bt.PULocationID
FROM workspace.taxi_weather.silver_trips AS bt
LEFT ANTI JOIN workspace.taxi_weather.silver_zone AS bz
ON bt.PULocationID = bz.LocationID
""",
    "Trips - Check for Null Dropoff LocationID": """
SELECT
DOLocationID
FROM workspace.taxi_weather.silver_trips
WHERE DOLocationID IS NULL
""",
    "Trips - Check for Unknown Dropoff LocationID": """
SELECT DISTINCT
bt.DOLocationID
FROM workspace.taxi_weather.silver_trips AS bt
LEFT ANTI JOIN workspace.taxi_weather.silver_zone AS bz
ON bt.DOLocationID = bz.LocationID
""",
    "Trips - Check for Null Pickup or Dropoff Datetime": """
SELECT
tpep_pickup_datetime,
tpep_dropoff_datetime
FROM workspace.taxi_weather.silver_trips
WHERE tpep_pickup_datetime IS NULL OR tpep_dropoff_datetime IS NULL
""",
    "Trips - Check for Invalid Pickup and Dropoff Datetime": """
SELECT
tpep_pickup_datetime,
tpep_dropoff_datetime
FROM workspace.taxi_weather.silver_trips
WHERE tpep_dropoff_datetime < tpep_pickup_datetime
""",
    "Weather - Check for Unknown Weather Code": """
SELECT DISTINCT
bz.weather_code
FROM workspace.taxi_weather.silver_weather AS bz
LEFT ANTI JOIN workspace.taxi_weather.silver_wc_dim AS sz
ON bz.weather_code = sz.weather_code
""",
    "Weather - Check for Null Weather Datetime": """
SELECT
weather_datetime,
COUNT(weather_datetime)
FROM workspace.taxi_weather.silver_weather
GROUP BY weather_datetime
HAVING weather_datetime IS NULL
""",
    "Weather - Check for Duplicate or Null Borough and Weather Datetime": """
SELECT
borough_id,
weather_datetime,
COUNT(*)
FROM workspace.taxi_weather.silver_weather
GROUP BY borough_id, weather_datetime
HAVING COUNT(*) > 1
OR borough_id IS NULL
OR weather_datetime IS NULL
""",
    "Zone - Check for Duplicate or Null LocationID": """
SELECT
LocationID,
COUNT(*)
FROM workspace.taxi_weather.silver_zone
GROUP BY LocationID
HAVING COUNT(*) > 1
OR LocationID IS NULL
""",
}

failed_checks = []

for check_name, query in dqc_queries.items():
    bad_record_count = spark.sql(query).count()
    if bad_record_count > 0:
        failed_checks.append(f"{check_name} ({bad_record_count} bad rows)")

if failed_checks:
    error_msg = "Pipeline Halted! Data Quality Checks Failed:\n" + "\n".join(failed_checks)
    raise RuntimeError(error_msg)
else:
    print("All Data Quality Checks passed.")